# ACE-Net Stage-2 — Consistency Discriminator  ·  PERSON E

Trains the fusion + MLP discriminator (paper spec: **50 epochs, patience 25**)
on the leakage-free actor-disjoint split, then evaluates Table 4.

**Requires only the TWO CREMA Stage-1 checkpoints** (`stage1_visual_crema.pt`,
`stage1_speech_text_crema.pt`) — the extractors are frozen. MELD checkpoints are
NOT used (no MELD forgeries).

Set Runtime → **T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone repo (feature branch)

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/acenet-training-pipeline
!git log --oneline -1

## 2. Install dependencies

In [ ]:
!pip -q install torch torchvision torchaudio transformers librosa pillow
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 3. Copy zip from Drive to local root, then unzip

Upload **`cremad_stage2.zip`** to Drive root. Must contain `CREMA-D/GENUINE_LastHalf`, `CREMA-D/GENUINE_FirstHalf`, `CREMA-D/FAKE_Paradigm1`, `CREMA-D/FAKE_Paradigm2`.

The zip is copied from `MyDrive` to the local Colab disk first — local I/O is much faster and avoids Drive-streaming stalls during training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob, shutil
DRIVE_ZIP = '/content/drive/MyDrive/cremad_stage2.zip'   # adjust if needed
LOCAL_ZIP = '/content/cremad_stage2.zip'
assert os.path.exists(DRIVE_ZIP), f'zip not found on Drive: {DRIVE_ZIP}'
print('copying zip to local root ...'); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
DST = '/content/Baseline_Training/data'
os.makedirs(DST, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z: z.extractall(DST)

# auto-relocate so data/CREMA-D and data/MELD sit at the expected level
for target in ['CREMA-D','MELD']:
    for h in [d for d in glob.glob(f'{DST}/**/{target}', recursive=True) if os.path.isdir(d)]:
        want=os.path.join(DST,target)
        if os.path.abspath(h)!=os.path.abspath(want): shutil.move(h, want)
for sub in ['GENUINE_LastHalf','GENUINE_FirstHalf','FAKE_Paradigm1','FAKE_Paradigm2']:
    loose=os.path.join(DST,sub)
    if os.path.isdir(loose):
        os.makedirs(os.path.join(DST,'CREMA-D'),exist_ok=True)
        shutil.move(loose, os.path.join(DST,'CREMA-D',sub))
print('data/ ->', os.listdir(DST))

## 3b. VERIFY extracted layout (stops here if the zip is wrong)

In [ ]:
import os
DST='/content/Baseline_Training/data'
required = [
    ('CREMA-D/GENUINE_LastHalf', os.path.isdir),
    ('CREMA-D/GENUINE_FirstHalf', os.path.isdir),
    ('CREMA-D/FAKE_Paradigm1', os.path.isdir),
    ('CREMA-D/FAKE_Paradigm2', os.path.isdir),
]
missing = [p for p,fn in required if not fn(os.path.join(DST,p))]
assert not missing, f'MISSING after unzip: {missing}\nRe-zip so these paths sit under data/ (zip the CREMA-D / MELD folder itself).'
for p,_ in required:
    full=os.path.join(DST,p)
    n=len(os.listdir(full)) if os.path.isdir(full) else 1
    print(f'  OK  {p}  ({n} entries)')
print('layout verified.')

## 4. Upload the two CREMA Stage-1 checkpoints

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)
from google.colab import files
print('upload stage1_visual_crema.pt and stage1_speech_text_crema.pt')
up = files.upload()
for name in up: os.replace(name, f'checkpoints/{name}')
have=set(os.listdir('checkpoints'))
need={'stage1_visual_crema.pt','stage1_speech_text_crema.pt'}
assert need<=have, f'missing checkpoints: {need-have}'
print('checkpoints present:', sorted(have))

## 5. Verify data + leakage-free split

Checks samples resolve, the split is actor-disjoint, and Stage-2 test actors
were never in Stage-1 training.

In [ ]:
import sys; sys.path.insert(0,'/content/Baseline_Training')
import random
from src.config import TrainConfig
from src.data import manifests
from src.data.splits import partition_by_actor
cfg=TrainConfig()
g,f=manifests.build_stage2_samples()
assert len(g)>0 and len(f)>0, 'genuine or fake samples missing -- check zip'
rng=random.Random(cfg.seed)
p1=[s for s in f if s.emotion=='crema_fake_p1']; p2=[s for s in f if s.emotion=='crema_fake_p2']
ne=min(len(p1),len(p2)); rng.shuffle(p1); rng.shuffle(p2)
fakes=p1[:ne]+p2[:ne]; rng.shuffle(fakes)
n=min(len(g),len(fakes)); rng.shuffle(g); chosen=g[:n]+fakes[:n]
tr,va,te=partition_by_actor(chosen, lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
cre=manifests.build_emotion_samples('crema')
s1tr,_,_=partition_by_actor(cre, lambda s:s.group_key,(0.8,0.1,0.1),cfg.seed)
assert min(len(tr),len(va),len(te))>0, 'empty split partition'
trg={s.group_key for s in tr}; teg={s.group_key for s in te}; s1trg={s.group_key for s in s1tr}
print(f'split train/val/test = {len(tr)}/{len(va)}/{len(te)}')
assert len(trg&teg)==0, 'train/test actor overlap!'
assert len(teg&s1trg)==0, 'Stage-2 test actor seen in Stage-1 train!'
print('VERIFIED: actor-disjoint, no cross-stage leak.')

## 6. Train Stage-2 (50 epochs, patience 25)

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.train_stage2 --batch-size 32 --epochs 50 --early-stop 25 --num-workers 2

## 7. Evaluate Table 4 — forgery detection by type

In [ ]:
!cd /content/Baseline_Training && PYTHONPATH=. python -m src.eval_stage2

## 8. Back up Stage-2 checkpoint to Drive

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/acenet_ckpts', exist_ok=True)
shutil.copy('checkpoints/stage2_acenet.pt','/content/drive/MyDrive/acenet_ckpts/stage2_acenet.pt')
print('saved stage2_acenet.pt to Drive')